[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/46_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding (draft, verify, resample)

*Inference & Decoding*
Implement one step of **speculative decoding**: a small draft model proposes $k$
tokens, the large target model verifies them all in a single forward pass, and
the accept/reject rule guarantees the output is distributed *exactly* as if you
had sampled from the target model directly.

### Signature
```python
def speculative_step(key, draft_tokens, draft_probs, target_probs):
    # draft_tokens: (k,) int   — tokens the draft model proposed
    # draft_probs:  (k, V)     — draft distribution at each of the k positions
    # target_probs: (k+1, V)   — target distribution at those k positions
    #                            PLUS one extra for the bonus token
    ...  # -> (accepted_tokens, n_accepted)
```

Return a `(k+1,)` int array padded with `-1` past the end, and the number of
draft tokens accepted.

### The rule, exactly
For each position $i$ in order, draw $u_i \sim U(0,1)$ and accept if

$$u_i < \min\left(1, \frac{p_{\text{target}}(x_i)}{p_{\text{draft}}(x_i)}\right)$$

- **Accept** → keep $x_i$, continue to $i+1$.
- **Reject** → stop, and sample a replacement from the **residual**
  $$p'(x) = \frac{\max\big(p_{\text{target}}(x) - p_{\text{draft}}(x),\ 0\big)}
  {\sum_x \max\big(p_{\text{target}}(x) - p_{\text{draft}}(x),\ 0\big)}$$
- **All $k$ accepted** → sample a **bonus** token from `target_probs[k]`.

So a step returns between 1 and $k+1$ tokens — never zero.

### Rules
- One uniform draw per position; use `jax.random.split`
- The residual must be renormalised, and clamped at zero *before* normalising
- Guard the degenerate case where the residual sums to 0
- Do not use a library implementation

### Why this is lossless — the property that matters
It is tempting to think this trades quality for speed. It does not, and the
proof is short: the probability of emitting token $x$ at a position is

$$\underbrace{p_d(x)\min\!\left(1, \tfrac{p_t(x)}{p_d(x)}\right)}_{\text{accepted}}
+ \underbrace{P(\text{reject})\cdot p'(x)}_{\text{resampled}} = p_t(x)$$

The first term is $\min(p_d(x), p_t(x))$. The residual contributes exactly the
missing $\max(p_t(x) - p_d(x), 0)$, and the rejection probability is precisely
its normalising constant — so they cancel and the total is $p_t(x)$.

**This is the whole point**, and it is why the residual must be
$\max(p_t - p_d, 0)$ renormalised and nothing else. Sampling the rejection from
$p_t$ directly is the classic bug: it over-weights tokens the draft already
liked, and quietly changes the output distribution. The tests below check the
resulting distribution statistically, not just the shapes.

### Where the speedup comes from
The target model runs once per *step*, not once per token, and a step emits up
to $k+1$ tokens. Decoding is memory-bandwidth bound — you pay to stream the
weights in regardless — so verifying $k$ tokens costs barely more than
generating one. The expected number of tokens per step rises with how well the
draft matches the target, which is why draft models are typically distilled from
their target.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def speculative_step(key, draft_tokens, draft_probs, target_probs):
    """One speculative decoding step.

    Args:
        key:          PRNG key
        draft_tokens: (k,) proposed token ids
        draft_probs:  (k, V) draft distribution at each position
        target_probs: (k+1, V) target distribution, including the bonus position

    Returns:
        (accepted, n_accepted)
          accepted   (k+1,) int32, emitted tokens then -1 padding
          n_accepted int, how many DRAFT tokens were accepted (0..k)
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

V, k = 4, 3
draft_probs = jnp.tile(jnp.array([0.7, 0.1, 0.1, 0.1]), (k, 1))
target_probs = jnp.tile(jnp.array([0.4, 0.3, 0.2, 0.1]), (k + 1, 1))
draft_tokens = jnp.array([0, 0, 0])

for seed in range(5):
    toks, n = speculative_step(jax.random.key(seed), draft_tokens,
                               draft_probs, target_probs)
    print(f"seed {seed}: accepted {n}/{k} draft tokens -> {toks.tolist()}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("speculative_decoding")

# hint("speculative_decoding")      # stuck? nudge without the answer
# solution("speculative_decoding")  # spoiler: the reference implementation